In [ ]:
!pip install -q tslearn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 387.9/387.9 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 141.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 34.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.64.0 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.64.0 which is incompatible.


In [ ]:
!pip install -q "chronos-forecasting>=2.1.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 122.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import itertools

import tslearn

from sklearn.preprocessing import StandardScaler, LabelEncoder
import umap

import warnings
#warnings.filterwarnings("ignore")

SEED = 42

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import torch.nn.functional as F

# Load data

In [ ]:
from tslearn.datasets import UCR_UEA_datasets

ds = UCR_UEA_datasets()
X_train, y_train, X_test, y_test = ds.load_dataset("LSST")

In [ ]:
print(f"Train X : {X_train.shape}  |  y : {y_train.shape}")
print(f"Test  X : {X_test.shape}  |  y : {y_test.shape}")
print(f"Classes : {sorted(np.unique(y_train).tolist())}")

Train X : (2459, 36, 6)  |  y : (2459,)
Test  X : (2466, 36, 6)  |  y : (2466,)
Classes : ['15', '16', '42', '52', '53', '6', '62', '64', '65', '67', '88', '90', '92', '95']


In [ ]:
N_TRAIN, N_STEPS, N_CH = X_train.shape
N_TEST = X_test.shape[0]
N_CLASSES = len(np.unique(y_train))

print(f"\nDataset summary")
print(f"  Samples  : {N_TRAIN} train  /  {N_TEST} test")
print(f"  Length   : {N_STEPS} time steps")


Dataset summary
  Samples  : 2459 train  /  2466 test
  Length   : 36 time steps


In [ ]:
class LSSTDataset(Dataset):

    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)  # (N, T, C)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


BATCH_SIZE = 64

le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

train_dataset = LSSTDataset(X_train, y_train)
test_dataset  = LSSTDataset(X_test,  y_test)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False,  num_workers=12, pin_memory=True)
test_loader   = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=12, pin_memory=True)

print(f"Train batches : {len(train_loader)}  ({len(train_dataset)} samples)")
print(f"Test  batches : {len(test_loader)}  ({len(test_dataset)} samples)")

Train batches : 39  (2459 samples)
Test  batches : 39  (2466 samples)


# Test mean pool over tokens per channel

In [ ]:
import torch
import torch.nn as nn

class Chronos2FeatureExtractor(nn.Module):
    def __init__(
        self,
        input_channels: int = 6,
        chronos_model_name: str = "amazon/chronos-2",
    ):
        super().__init__()
        self.input_channels = input_channels

        chronos_device = "cuda" if torch.cuda.is_available() else "cpu"
        chronos_dtype = torch.bfloat16 if chronos_device == "cuda" else torch.float32

        # Make sure to import Chronos2Pipeline where appropriate in your file
        self.pipeline = Chronos2Pipeline.from_pretrained(
            chronos_model_name,
            device_map=chronos_device,
            dtype=chronos_dtype,
        )

        # Freeze Chronos
        for param in self.pipeline.model.parameters():
            param.requires_grad = False

    @torch.no_grad()
    def extract_embeddings(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim != 3:
            raise ValueError(f"Expected (batch, timesteps, channels), got {tuple(x.shape)}")

        x_permuted = x.permute(0, 2, 1).contiguous()
        chronos_inputs = [sample.cpu() for sample in x_permuted]

        # embeddings is a list of tensors, one for each sample in the batch
        embeddings, _ = self.pipeline.embed(chronos_inputs)

        pooled_features = []
        for sample_embedding in embeddings:
            # sample_embedding shape: (channels, num_tokens, embedding_dim)
            # Example: (6, T', 768)

            # Mean-pool over the token dimension (dim=1)
            # Resulting shape: (channels, embedding_dim) -> e.g., (6, 768)
            mean_pooled = sample_embedding.mean(dim=1)

            # Flatten out the channels and embedding_dim into a single vector
            # Resulting shape: (channels * embedding_dim,) -> e.g., (4608,)
            pooled = mean_pooled.reshape(-1)

            pooled_features.append(pooled)

        return torch.stack(pooled_features, dim=0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.extract_embeddings(x).to(dtype=torch.float32)

In [ ]:
def get_all_embeddings(loader, model, device):
    all_features = []
    all_labels = []

    print("Extracting embeddings :")
    for xb, yb in loader:
        xb = xb.to(device)
        features = model(xb)

        all_features.append(features.cpu().numpy())
        all_labels.append(yb.cpu().numpy())

    return np.vstack(all_features), np.concatenate(all_labels)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

extractor = Chronos2FeatureExtractor(
    input_channels=N_CH,
    chronos_model_name="amazon/chronos-2",
).to(device)

extractor.eval()

# Extract embeddings from train and test
print("Training Set")
X_train_emb, y_train_np = get_all_embeddings(train_loader, extractor, device)

print("Testing Set")
X_test_emb, y_test_np = get_all_embeddings(test_loader, extractor, device)

print(f"\nFinal Training Matrix Shape : {X_train_emb.shape}")
print(f"Final Testing Matrix Shape  : {X_test_emb.shape}")

Training Set
Extracting embeddings :
Testing Set
Extracting embeddings :

Final Training Matrix Shape : (2459, 4608)
Final Testing Matrix Shape  : (2466, 4608)


In [ ]:
import numpy as np
import scipy.stats as sp_stats
from scipy.signal import find_peaks

def find_time_to_fractions(fluxes, fractions, forward=True):
    max_time = np.argmax(fluxes)
    max_flux = fluxes[max_time]

    result = np.zeros(len(fractions))
    frac_idx = 0
    offset = 0

    while True:
        offset += 1
        if forward:
            new_time = max_time + offset
            if new_time >= len(fluxes):
                # Hit the end of array, fill remaining with max offset
                result[frac_idx:] = offset
                break
        else:
            new_time = max_time - offset
            if new_time < 0:
                result[frac_idx:] = offset
                break

        test_flux = fluxes[new_time]
        while test_flux < max_flux * fractions[frac_idx]:
            result[frac_idx] = offset
            frac_idx += 1
            if frac_idx == len(fractions):
                break

        if frac_idx == len(fractions):
            break

    return result

def compute_handcrafted_features(X):
    X = np.asarray(X, dtype=np.float32)

    if X.ndim != 3:
        raise ValueError(f"Expected (N, T, C), got {X.shape}")

    N, T, C = X.shape

    # Define band indices (assuming u=0, g=1, r=2, i=3, z=4, y=5)
    b_g = 1 # blue band
    b_i = 3 # reference band
    b_y = 5 # red band

    feats = []

    for i in range(N):
        sample_feats = []
        X_i = X[i]

        # Base Features

        # Per-channel stats: 10 features per channel
        for c in range(C):
            ts = X_i[:, c]
            sample_feats.extend([
                np.mean(ts),
                np.std(ts),
                np.min(ts),
                np.max(ts),
                np.ptp(ts),  # range
                float(sp_stats.skew(ts)),
                float(sp_stats.kurtosis(ts)),
                np.polyfit(np.arange(T), ts, 1)[0],  # slope
                ts[0],
                ts[-1],
            ])

        # Cross-channel correlations: C*(C-1)/2 = 15 if C=6
        for c1 in range(C):
            for c2 in range(c1 + 1, C):
                r = np.corrcoef(X_i[:, c1], X_i[:, c2])[0, 1]
                sample_feats.append(r if np.isfinite(r) else 0.0)

        # Amplitude ratios: also 15 if C=6
        amplitudes = [np.ptp(X_i[:, c]) for c in range(C)]
        for c1 in range(C):
            for c2 in range(c1 + 1, C):
                ratio = amplitudes[c1] / (amplitudes[c2] + 1e-8)
                sample_feats.append(np.log1p(ratio))

        # Avocado Features
        avocado_feats = []

        max_flux_i = np.max(X_i[:, b_i])
        min_flux_i = np.min(X_i[:, b_i])
        max_flux_g = np.max(X_i[:, b_g])
        min_flux_g = np.min(X_i[:, b_g])
        max_flux_y = np.max(X_i[:, b_y])
        min_flux_y = np.min(X_i[:, b_y])

        # Magnitude & Flux ratios
        max_mag = -2.5 * np.log10(np.abs(max_flux_i) + 1e-8)
        pos_flux_ratio = max_flux_i / (max_flux_i - min_flux_i + 1e-8)

        max_flux_ratio_red = np.abs(max_flux_y) / (np.abs(max_flux_y) + np.abs(max_flux_i) + 1e-8)
        max_flux_ratio_blue = np.abs(max_flux_g) / (np.abs(max_flux_i) + np.abs(max_flux_g) + 1e-8)

        min_flux_ratio_red = np.abs(min_flux_y) / (np.abs(min_flux_y) + np.abs(min_flux_i) + 1e-8)
        min_flux_ratio_blue = np.abs(min_flux_g) / (np.abs(min_flux_i) + np.abs(min_flux_g) + 1e-8)

        avocado_feats.extend([
            max_mag, pos_flux_ratio,
            max_flux_ratio_red, max_flux_ratio_blue,
            min_flux_ratio_red, min_flux_ratio_blue
        ])

        # Timing differences
        max_dt = np.argmax(X_i[:, b_y]) - np.argmax(X_i[:, b_g])
        avocado_feats.append(max_dt)

        # Width Integrals
        positive_width = np.sum(np.clip(X_i[:, b_i], 0, None)) / (max_flux_i + 1e-8)
        negative_width = np.sum(np.clip(X_i[:, b_i], None, 0)) / (min_flux_i - 1e-8)
        avocado_feats.extend([positive_width, negative_width])

        # Time to fractions of maximum
        fractions = [0.5, 0.2]
        fwd_times_i = find_time_to_fractions(X_i[:, b_i], fractions, forward=True)
        bwd_times_i = find_time_to_fractions(X_i[:, b_i], fractions, forward=False)
        fwd_times_g = find_time_to_fractions(X_i[:, b_g], fractions, forward=True)
        bwd_times_g = find_time_to_fractions(X_i[:, b_g], fractions, forward=False)
        fwd_times_y = find_time_to_fractions(X_i[:, b_y], fractions, forward=True)
        bwd_times_y = find_time_to_fractions(X_i[:, b_y], fractions, forward=False)

        # 0.5 and 0.2 ratios (forward)
        fwd_05_red = fwd_times_y[0] / (fwd_times_y[0] + fwd_times_i[0] + 1e-8)
        fwd_05_blue = fwd_times_g[0] / (fwd_times_g[0] + fwd_times_i[0] + 1e-8)
        fwd_02_red = fwd_times_y[1] / (fwd_times_y[1] + fwd_times_i[1] + 1e-8)
        fwd_02_blue = fwd_times_g[1] / (fwd_times_g[1] + fwd_times_i[1] + 1e-8)

        # 0.5 and 0.2 ratios (backward)
        bwd_05_red = bwd_times_y[0] / (bwd_times_y[0] + bwd_times_i[0] + 1e-8)
        bwd_05_blue = bwd_times_g[0] / (bwd_times_g[0] + bwd_times_i[0] + 1e-8)
        bwd_02_red = bwd_times_y[1] / (bwd_times_y[1] + bwd_times_i[1] + 1e-8)
        bwd_02_blue = bwd_times_g[1] / (bwd_times_g[1] + bwd_times_i[1] + 1e-8)

        avocado_feats.extend([
            fwd_times_i[0], fwd_times_i[1], fwd_05_red, fwd_05_blue, fwd_02_red, fwd_02_blue,
            bwd_times_i[0], bwd_times_i[1], bwd_05_red, bwd_05_blue, bwd_02_red, bwd_02_blue
        ])

        # Percentile differences
        percentiles = [10, 30, 50, 70, 90]
        all_frac_percentiles = []
        for p in percentiles:
            frac_percentiles = []
            for c in range(C):
                p_val = np.percentile(X_i[:, c], p)
                mx = np.max(X_i[:, c])
                mn = np.min(X_i[:, c])
                frac_percentiles.append((p_val - mn) / (mx - mn + 1e-8))
            all_frac_percentiles.append(np.nanmedian(frac_percentiles))

        avocado_feats.extend([
            all_frac_percentiles[0] - all_frac_percentiles[2], # 10 - 50
            all_frac_percentiles[1] - all_frac_percentiles[2], # 30 - 50
            all_frac_percentiles[3] - all_frac_percentiles[2], # 70 - 50
            all_frac_percentiles[4] - all_frac_percentiles[2]  # 90 - 50
        ])

        # Secondary Peak Fraction
        peak_fracs_2 = []
        for c in range(C):
            flux = X_i[:, c]
            peaks, props = find_peaks(flux, height=np.max(np.abs(flux))/5.0)
            heights = np.sort(props["peak_heights"])[::-1]
            if len(heights) > 1:
                peak_fracs_2.append(heights[1] / heights[0])
            else:
                peak_fracs_2.append(0.0)
        peak_frac_2 = np.nanmedian(peak_fracs_2) if len(peak_fracs_2) > 0 else 0.0
        avocado_feats.append(peak_frac_2)

        # Append avocado feats to main list
        sample_feats.extend(avocado_feats)
        feats.append(sample_feats)

    feats = np.array(feats, dtype=np.float32)
    feats = np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0)
    return feats


def clean(arr):
    return np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)

In [ ]:
hc_train = compute_handcrafted_features(X_train)
hc_test = compute_handcrafted_features(X_test)

print(f"\nHandcrafted train shape: {hc_train.shape}")
print(f"Handcrafted test  shape: {hc_test.shape}")

concat_train_new = clean(np.concatenate([X_train_emb, hc_train], axis=1))
concat_test_new = clean(np.concatenate([X_test_emb, hc_test], axis=1))

/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]



Handcrafted train shape: (2459, 116)
Handcrafted test  shape: (2466, 116)


In [ ]:
classes = np.unique(y_train_np)
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_np
)
class_weight_dict = dict(zip(classes, weights))
print("class_weight_dict:", class_weight_dict)

sample_weights = np.array([class_weight_dict[y] for y in y_train_np])

seeds = [42] # 52, 62, 72, 82
models = []

for seed in seeds:
    clf = XGBClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.5,
        random_state=seed,
        tree_method='hist',
        eval_metric='mlogloss'
    )

    clf.fit(concat_train_new, y_train_np, sample_weight=sample_weights)
    models.append(clf)

proba_list = [clf.predict_proba(concat_test_new) for clf in models]
mean_proba = np.mean(proba_list, axis=0)
y_pred = mean_proba.argmax(axis=1)

print("Accuracy:", accuracy_score(y_test_np, y_pred))
print("Balanced Accuracy:", balanced_accuracy_score(y_test_np, y_pred))
print("Macro F1:", f1_score(y_test_np, y_pred, average="macro"))
print(classification_report(y_test_np, y_pred, digits=4, zero_division=0))

class_weight_dict: {np.int64(0): np.float64(1.4279907084785133), np.int64(1): np.float64(0.6505291005291005), np.int64(2): np.float64(0.46100487439070115), np.int64(3): np.float64(2.8329493087557602), np.int64(4): np.float64(25.091836734693878), np.int64(5): np.float64(5.165966386554622), np.int64(6): np.float64(1.1479925303454714), np.int64(7): np.float64(7.636645962732919), np.int64(8): np.float64(0.5611592879963487), np.int64(9): np.float64(2.582983193277311), np.int64(10): np.float64(1.463690476190476), np.int64(11): np.float64(0.22605258319544033), np.int64(12): np.float64(2.2810760667903525), np.int64(13): np.float64(3.4439775910364148)}
Accuracy: 0.7141119221411192
Balanced Accuracy: 0.5520510790304505
Macro F1: 0.5726326867763717
              precision    recall  f1-score   support

           0     0.6800    0.5484    0.6071       124
           1     0.9813    0.9704    0.9758       270
           2     0.5822    0.4634    0.5160       382
           3     1.0000    0.0159  

# REG extract for embedding

In [ ]:
class Chronos2FeatureExtractor(nn.Module):
    def __init__(
        self,
        input_channels: int = 6,
        chronos_model_name: str = "amazon/chronos-2",
    ):
        super().__init__()
        self.input_channels = input_channels

        chronos_device = "cuda" if torch.cuda.is_available() else "cpu"
        chronos_dtype = torch.bfloat16 if chronos_device == "cuda" else torch.float32

        self.pipeline = Chronos2Pipeline.from_pretrained(
            chronos_model_name,
            device_map=chronos_device,
            dtype=chronos_dtype,
        )

        # Freeze Chronos
        for param in self.pipeline.model.parameters():
            param.requires_grad = False

    @torch.no_grad()
    def extract_embeddings(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim != 3:
            raise ValueError(f"Expected (batch, timesteps, channels), got {tuple(x.shape)}")

        x_permuted = x.permute(0, 2, 1).contiguous()
        chronos_inputs = [sample.cpu() for sample in x_permuted]

        embeddings, _ = self.pipeline.embed(chronos_inputs)

        pooled_features = []
        for sample_embedding in embeddings:
            # reg_tokens = sample_embedding[:, -2, :]
            # pooled = torch.max(reg_tokens, dim=0).values

            reg_tokens = sample_embedding[:, -2, :]
            pooled = reg_tokens.reshape(-1)

            pooled_features.append(pooled)

        return torch.stack(pooled_features, dim=0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.extract_embeddings(x).to(dtype=torch.float32)

In [ ]:
def get_all_embeddings(loader, model, device):
    all_features = []
    all_labels = []

    print("Extracting embeddings :")
    for xb, yb in loader:
        xb = xb.to(device)
        features = model(xb)

        all_features.append(features.cpu().numpy())
        all_labels.append(yb.cpu().numpy())

    return np.vstack(all_features), np.concatenate(all_labels)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

extractor = Chronos2FeatureExtractor(
    input_channels=N_CH,
    chronos_model_name="amazon/chronos-2",
).to(device)

extractor.eval()

# Extract embeddings from train and test
print("Training Set")
X_train_emb, y_train_np = get_all_embeddings(train_loader, extractor, device)

print("Testing Set")
X_test_emb, y_test_np = get_all_embeddings(test_loader, extractor, device)

print(f"\nFinal Training Matrix Shape : {X_train_emb.shape}")
print(f"Final Testing Matrix Shape  : {X_test_emb.shape}")

Training Set
Extracting embeddings :
Testing Set
Extracting embeddings :

Final Training Matrix Shape : (2459, 4608)
Final Testing Matrix Shape  : (2466, 4608)


In [ ]:
ridge_clf = RidgeClassifierCV(alphas=np.logspace(-3, 3, 10))
ridge_clf.fit(X_train_emb, y_train_np)

RidgeClassifierCV(alphas=array([1.00000000e-03, 4.64158883e-03, 2.15443469e-02, 1.00000000e-01,
       4.64158883e-01, 2.15443469e+00, 1.00000000e+01, 4.64158883e+01,
       2.15443469e+02, 1.00000000e+03]))

In [ ]:
ridge_preds = ridge_clf.predict(X_test_emb)
print(classification_report(y_test_np, ridge_preds))

              precision    recall  f1-score   support

           0       0.91      0.23      0.37       124
           1       0.81      0.96      0.88       270
           2       0.53      0.37      0.43       382
           3       0.00      0.00      0.00        63
           4       0.00      0.00      0.00         7
           5       0.50      0.03      0.05        35
           6       0.17      0.03      0.05       153
           7       0.00      0.00      0.00        24
           8       0.66      0.84      0.74       313
           9       0.00      0.00      0.00        68
          10       0.93      0.95      0.94       121
          11       0.55      0.88      0.67       777
          12       0.91      0.51      0.65        77
          13       0.50      0.02      0.04        52

    accuracy                           0.62      2466
   macro avg       0.46      0.34      0.34      2466
weighted avg       0.57      0.62      0.56      2466



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeClassifierCV, LogisticRegressionCV
from sklearn.svm import LinearSVC, SVC
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report

models = {
    "ridge": make_pipeline(
        StandardScaler(),
        RidgeClassifierCV(alphas=np.logspace(-3, 3, 13))
    ),
    "logreg": make_pipeline(
        StandardScaler(),
        LogisticRegressionCV(
            Cs=10,
            cv=5,
            max_iter=5000,
            multi_class="multinomial",
            class_weight="balanced",
            n_jobs=-1
        )
    ),
}

for name, clf in models.items():
    clf.fit(X_train_emb, y_train_np)
    y_pred = clf.predict(X_test_emb)
    print(f"\n=== {name} ===")
    print("Accuracy:", accuracy_score(y_test_np, y_pred))
    print(classification_report(y_test_np, y_pred, digits=4))


=== ridge ===
Accuracy: 0.6321978913219789
              precision    recall  f1-score   support

           0     0.7619    0.3871    0.5134       124
           1     0.8462    0.8963    0.8705       270
           2     0.4747    0.4660    0.4703       382
           3     0.0000    0.0000    0.0000        63
           4     0.0000    0.0000    0.0000         7
           5     0.8000    0.1143    0.2000        35
           6     0.2979    0.0915    0.1400       153
           7     0.0000    0.0000    0.0000        24
           8     0.6935    0.8530    0.7650       313
           9     0.0000    0.0000    0.0000        68
          10     0.9417    0.9339    0.9378       121
          11     0.5802    0.8288    0.6826       777
          12     0.6866    0.5974    0.6389        77
          13     0.6000    0.0577    0.1053        52

    accuracy                         0.6322      2466
   macro avg     0.4773    0.3733    0.3803      2466
weighted avg     0.5854    0.6322   

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/l


=== logreg ===
Accuracy: 0.5977291159772912
              precision    recall  f1-score   support

           0     0.4812    0.5161    0.4981       124
           1     0.9091    0.9259    0.9174       270
           2     0.4557    0.4712    0.4633       382
           3     0.1250    0.0952    0.1081        63
           4     0.0000    0.0000    0.0000         7
           5     0.4091    0.2571    0.3158        35
           6     0.2143    0.2745    0.2407       153
           7     0.1250    0.0833    0.1000        24
           8     0.7471    0.8115    0.7779       313
           9     0.1266    0.1471    0.1361        68
          10     0.9256    0.9256    0.9256       121
          11     0.6657    0.6178    0.6409       777
          12     0.7571    0.6883    0.7211        77
          13     0.2449    0.2308    0.2376        52

    accuracy                         0.5977      2466
   macro avg     0.4419    0.4318    0.4345      2466
weighted avg     0.6001    0.5977  

In [ ]:
import numpy as np
import lightgbm as lgb
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)
class_weight_dict = dict(zip(classes, weights))
print("class_weight_dict:", class_weight_dict)

seeds = [42] # 52, 62, 72, 82
models = []

for seed in seeds:
    clf = lgb.LGBMClassifier(
        objective="multiclass",
        num_class=len(classes),
        boosting_type="gbdt",
        n_estimators=200,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=1.0,
        class_weight=class_weight_dict,
        random_state=seed,
        n_jobs=-1,
        verbosity=-1,
    )

    clf.fit(X_train_emb, y_train_np)
    models.append(clf)

proba_list = [clf.predict_proba(X_test_emb) for clf in models]
mean_proba = np.mean(proba_list, axis=0)
y_pred = mean_proba.argmax(axis=1)

print("Accuracy:", accuracy_score(y_test_np, y_pred))
print("Balanced Accuracy:", balanced_accuracy_score(y_test_np, y_pred))
print("Macro F1:", f1_score(y_test_np, y_pred, average="macro"))
print(classification_report(y_test_np, y_pred, digits=4, zero_division=0))

class_weight_dict: {np.int64(0): np.float64(1.4279907084785133), np.int64(1): np.float64(0.6505291005291005), np.int64(2): np.float64(0.46100487439070115), np.int64(3): np.float64(2.8329493087557602), np.int64(4): np.float64(25.091836734693878), np.int64(5): np.float64(5.165966386554622), np.int64(6): np.float64(1.1479925303454714), np.int64(7): np.float64(7.636645962732919), np.int64(8): np.float64(0.5611592879963487), np.int64(9): np.float64(2.582983193277311), np.int64(10): np.float64(1.463690476190476), np.int64(11): np.float64(0.22605258319544033), np.int64(12): np.float64(2.2810760667903525), np.int64(13): np.float64(3.4439775910364148)}
Accuracy: 0.6492295214922952
Balanced Accuracy: 0.38805917559241915
Macro F1: 0.390285698707205
              precision    recall  f1-score   support

           0     0.6753    0.4194    0.5174       124
           1     0.9046    0.9481    0.9259       270
           2     0.5428    0.3822    0.4485       382
           3     0.0000    0.0000  

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predict

## Adding handcrafted features

In [ ]:
import scipy.stats as sp_stats

def compute_handcrafted_features(X):
    X = np.asarray(X, dtype=np.float32)

    if X.ndim != 3:
        raise ValueError(f"Expected (N, T, C), got {X.shape}")

    N, T, C = X.shape
    feats = []

    for i in range(N):
        sample_feats = []

        # Per-channel stats: 10 features per channel
        for c in range(C):
            ts = X[i, :, c]
            sample_feats.extend([
                np.mean(ts),
                np.std(ts),
                np.min(ts),
                np.max(ts),
                np.ptp(ts),  # range
                float(sp_stats.skew(ts)),
                float(sp_stats.kurtosis(ts)),
                np.polyfit(np.arange(T), ts, 1)[0],  # slope
                ts[0],
                ts[-1],
            ])

        # Cross-channel correlations: C*(C-1)/2 = 15 if C=6
        for c1 in range(C):
            for c2 in range(c1 + 1, C):
                r = np.corrcoef(X[i, :, c1], X[i, :, c2])[0, 1]
                sample_feats.append(r if np.isfinite(r) else 0.0)

        # Amplitude ratios: also 15 if C=6
        amplitudes = [np.ptp(X[i, :, c]) for c in range(C)]
        for c1 in range(C):
            for c2 in range(c1 + 1, C):
                ratio = amplitudes[c1] / (amplitudes[c2] + 1e-8)
                sample_feats.append(np.log1p(ratio))

        feats.append(sample_feats)

    feats = np.array(feats, dtype=np.float32)
    feats = np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0)
    return feats


def clean(arr):
    return np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)

In [ ]:
hc_train = compute_handcrafted_features(X_train)
hc_test = compute_handcrafted_features(X_test)

print(f"\nHandcrafted train shape: {hc_train.shape}")
print(f"Handcrafted test  shape: {hc_test.shape}")

concat_train = clean(np.concatenate([X_train_emb, hc_train], axis=1))
concat_test = clean(np.concatenate([X_test_emb, hc_test], axis=1))

/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]



Handcrafted train shape: (2459, 90)
Handcrafted test  shape: (2466, 90)


In [ ]:
classes = np.unique(y_train)
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)
class_weight_dict = dict(zip(classes, weights))
print("class_weight_dict:", class_weight_dict)

seeds = [42] # 52, 62, 72, 82
models = []

for seed in seeds:
    clf = lgb.LGBMClassifier(
        objective="multiclass",
        num_class=len(classes),
        boosting_type="gbdt",
        n_estimators=200,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=1.0,
        class_weight=class_weight_dict,
        random_state=seed,
        n_jobs=-1,
        verbosity=-1,
    )

    clf.fit(concat_train, y_train_np)
    models.append(clf)

proba_list = [clf.predict_proba(concat_test) for clf in models]
mean_proba = np.mean(proba_list, axis=0)
y_pred = mean_proba.argmax(axis=1)

print("Accuracy:", accuracy_score(y_test_np, y_pred))
print("Balanced Accuracy:", balanced_accuracy_score(y_test_np, y_pred))
print("Macro F1:", f1_score(y_test_np, y_pred, average="macro"))
print(classification_report(y_test_np, y_pred, digits=4, zero_division=0))

class_weight_dict: {np.int64(0): np.float64(1.4279907084785133), np.int64(1): np.float64(0.6505291005291005), np.int64(2): np.float64(0.46100487439070115), np.int64(3): np.float64(2.8329493087557602), np.int64(4): np.float64(25.091836734693878), np.int64(5): np.float64(5.165966386554622), np.int64(6): np.float64(1.1479925303454714), np.int64(7): np.float64(7.636645962732919), np.int64(8): np.float64(0.5611592879963487), np.int64(9): np.float64(2.582983193277311), np.int64(10): np.float64(1.463690476190476), np.int64(11): np.float64(0.22605258319544033), np.int64(12): np.float64(2.2810760667903525), np.int64(13): np.float64(3.4439775910364148)}
Accuracy: 0.7088402270884022
Balanced Accuracy: 0.5225483979832278
Macro F1: 0.5380925582721087
              precision    recall  f1-score   support

           0     0.6735    0.5323    0.5946       124
           1     0.9850    0.9704    0.9776       270
           2     0.5586    0.4869    0.5203       382
           3     0.0000    0.0000  

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
classes = np.unique(y_train)
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)
class_weight_dict = dict(zip(classes, weights))
print("class_weight_dict:", class_weight_dict)

seeds = [42, 52, 62] # 52, 62, 72, 82
models = []

for seed in seeds:
    clf = lgb.LGBMClassifier(
        objective="multiclass",
        num_class=len(classes),
        boosting_type="gbdt",
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=1.0,
        class_weight=class_weight_dict,
        random_state=seed,
        n_jobs=-1,
        verbosity=-1,
    )

    clf.fit(concat_train, y_train_np)
    models.append(clf)

proba_list = [clf.predict_proba(concat_test) for clf in models]
mean_proba = np.mean(proba_list, axis=0)
y_pred = mean_proba.argmax(axis=1)

print("Accuracy:", accuracy_score(y_test_np, y_pred))
print("Balanced Accuracy:", balanced_accuracy_score(y_test_np, y_pred))
print("Macro F1:", f1_score(y_test_np, y_pred, average="macro"))
print(classification_report(y_test_np, y_pred, digits=4, zero_division=0))

class_weight_dict: {np.int64(0): np.float64(1.4279907084785133), np.int64(1): np.float64(0.6505291005291005), np.int64(2): np.float64(0.46100487439070115), np.int64(3): np.float64(2.8329493087557602), np.int64(4): np.float64(25.091836734693878), np.int64(5): np.float64(5.165966386554622), np.int64(6): np.float64(1.1479925303454714), np.int64(7): np.float64(7.636645962732919), np.int64(8): np.float64(0.5611592879963487), np.int64(9): np.float64(2.582983193277311), np.int64(10): np.float64(1.463690476190476), np.int64(11): np.float64(0.22605258319544033), np.int64(12): np.float64(2.2810760667903525), np.int64(13): np.float64(3.4439775910364148)}


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Accuracy: 0.7116788321167883
Balanced Accuracy: 0.5235064754749569
Macro F1: 0.5404930991171866
              precision    recall  f1-score   support

           0     0.6915    0.5242    0.5963       124
           1     0.9850    0.9704    0.9776       270
           2     0.5719    0.4791    0.5214       382
           3     0.0000    0.0000    0.0000        63
           4     0.8571    0.8571    0.8571         7
           5     0.6364    0.2000    0.3043        35
           6     0.5000    0.2941    0.3704       153
           7     0.0000    0.0000    0.0000        24
           8     0.7575    0.8882    0.8176       313
           9     0.6471    0.1618    0.2588        68
          10     0.9344    0.9421    0.9383       121
          11     0.6512    0.9035    0.7569       777
          12     0.9259    0.9740    0.9494        77
          13     0.5833    0.1346    0.2188        52

    accuracy                         0.7117      2466
   macro avg     0.6244    0.5235    0

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
classes = np.unique(y_train)
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)
class_weight_dict = dict(zip(classes, weights))
print("class_weight_dict:", class_weight_dict)

seeds = [42, 52, 62] # 52, 62, 72, 82
models = []

for seed in seeds:
    clf = lgb.LGBMClassifier(
        objective="multiclass",
        num_class=len(classes),
        boosting_type="gbdt",
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=1.0,
        class_weight=class_weight_dict,
        random_state=seed,
        n_jobs=-1,
        verbosity=-1,
    )

    clf.fit(concat_train, y_train_np)
    models.append(clf)

proba_list = [clf.predict_proba(concat_test) for clf in models]
mean_proba = np.mean(proba_list, axis=0)
y_pred = mean_proba.argmax(axis=1)

print("Accuracy:", accuracy_score(y_test_np, y_pred))
print("Balanced Accuracy:", balanced_accuracy_score(y_test_np, y_pred))
print("Macro F1:", f1_score(y_test_np, y_pred, average="macro"))
print(classification_report(y_test_np, y_pred, digits=4, zero_division=0))

class_weight_dict: {np.int64(0): np.float64(1.4279907084785133), np.int64(1): np.float64(0.6505291005291005), np.int64(2): np.float64(0.46100487439070115), np.int64(3): np.float64(2.8329493087557602), np.int64(4): np.float64(25.091836734693878), np.int64(5): np.float64(5.165966386554622), np.int64(6): np.float64(1.1479925303454714), np.int64(7): np.float64(7.636645962732919), np.int64(8): np.float64(0.5611592879963487), np.int64(9): np.float64(2.582983193277311), np.int64(10): np.float64(1.463690476190476), np.int64(11): np.float64(0.22605258319544033), np.int64(12): np.float64(2.2810760667903525), np.int64(13): np.float64(3.4439775910364148)}


## Adding more handcrafted features

In [ ]:
import numpy as np
import scipy.stats as sp_stats
from scipy.signal import find_peaks

def find_time_to_fractions(fluxes, fractions, forward=True):
    max_time = np.argmax(fluxes)
    max_flux = fluxes[max_time]

    result = np.zeros(len(fractions))
    frac_idx = 0
    offset = 0

    while True:
        offset += 1
        if forward:
            new_time = max_time + offset
            if new_time >= len(fluxes):
                # Hit the end of array, fill remaining with max offset
                result[frac_idx:] = offset
                break
        else:
            new_time = max_time - offset
            if new_time < 0:
                result[frac_idx:] = offset
                break

        test_flux = fluxes[new_time]
        while test_flux < max_flux * fractions[frac_idx]:
            result[frac_idx] = offset
            frac_idx += 1
            if frac_idx == len(fractions):
                break

        if frac_idx == len(fractions):
            break

    return result

def compute_handcrafted_features(X):
    X = np.asarray(X, dtype=np.float32)

    if X.ndim != 3:
        raise ValueError(f"Expected (N, T, C), got {X.shape}")

    N, T, C = X.shape

    # Define band indices (assuming u=0, g=1, r=2, i=3, z=4, y=5)
    b_g = 1 # blue band
    b_i = 3 # reference band
    b_y = 5 # red band

    feats = []

    for i in range(N):
        sample_feats = []
        X_i = X[i]

        # Base Features

        # Per-channel stats: 10 features per channel
        for c in range(C):
            ts = X_i[:, c]
            sample_feats.extend([
                np.mean(ts),
                np.std(ts),
                np.min(ts),
                np.max(ts),
                np.ptp(ts),  # range
                float(sp_stats.skew(ts)),
                float(sp_stats.kurtosis(ts)),
                np.polyfit(np.arange(T), ts, 1)[0],  # slope
                ts[0],
                ts[-1],
            ])

        # Cross-channel correlations: C*(C-1)/2 = 15 if C=6
        for c1 in range(C):
            for c2 in range(c1 + 1, C):
                r = np.corrcoef(X_i[:, c1], X_i[:, c2])[0, 1]
                sample_feats.append(r if np.isfinite(r) else 0.0)

        # Amplitude ratios: also 15 if C=6
        amplitudes = [np.ptp(X_i[:, c]) for c in range(C)]
        for c1 in range(C):
            for c2 in range(c1 + 1, C):
                ratio = amplitudes[c1] / (amplitudes[c2] + 1e-8)
                sample_feats.append(np.log1p(ratio))

        # Avocado Features
        avocado_feats = []

        max_flux_i = np.max(X_i[:, b_i])
        min_flux_i = np.min(X_i[:, b_i])
        max_flux_g = np.max(X_i[:, b_g])
        min_flux_g = np.min(X_i[:, b_g])
        max_flux_y = np.max(X_i[:, b_y])
        min_flux_y = np.min(X_i[:, b_y])

        # Magnitude & Flux ratios
        max_mag = -2.5 * np.log10(np.abs(max_flux_i) + 1e-8)
        pos_flux_ratio = max_flux_i / (max_flux_i - min_flux_i + 1e-8)

        max_flux_ratio_red = np.abs(max_flux_y) / (np.abs(max_flux_y) + np.abs(max_flux_i) + 1e-8)
        max_flux_ratio_blue = np.abs(max_flux_g) / (np.abs(max_flux_i) + np.abs(max_flux_g) + 1e-8)

        min_flux_ratio_red = np.abs(min_flux_y) / (np.abs(min_flux_y) + np.abs(min_flux_i) + 1e-8)
        min_flux_ratio_blue = np.abs(min_flux_g) / (np.abs(min_flux_i) + np.abs(min_flux_g) + 1e-8)

        avocado_feats.extend([
            max_mag, pos_flux_ratio,
            max_flux_ratio_red, max_flux_ratio_blue,
            min_flux_ratio_red, min_flux_ratio_blue
        ])

        # Timing differences
        max_dt = np.argmax(X_i[:, b_y]) - np.argmax(X_i[:, b_g])
        avocado_feats.append(max_dt)

        # Width Integrals
        positive_width = np.sum(np.clip(X_i[:, b_i], 0, None)) / (max_flux_i + 1e-8)
        negative_width = np.sum(np.clip(X_i[:, b_i], None, 0)) / (min_flux_i - 1e-8)
        avocado_feats.extend([positive_width, negative_width])

        # Time to fractions of maximum
        fractions = [0.5, 0.2]
        fwd_times_i = find_time_to_fractions(X_i[:, b_i], fractions, forward=True)
        bwd_times_i = find_time_to_fractions(X_i[:, b_i], fractions, forward=False)
        fwd_times_g = find_time_to_fractions(X_i[:, b_g], fractions, forward=True)
        bwd_times_g = find_time_to_fractions(X_i[:, b_g], fractions, forward=False)
        fwd_times_y = find_time_to_fractions(X_i[:, b_y], fractions, forward=True)
        bwd_times_y = find_time_to_fractions(X_i[:, b_y], fractions, forward=False)

        # 0.5 and 0.2 ratios (forward)
        fwd_05_red = fwd_times_y[0] / (fwd_times_y[0] + fwd_times_i[0] + 1e-8)
        fwd_05_blue = fwd_times_g[0] / (fwd_times_g[0] + fwd_times_i[0] + 1e-8)
        fwd_02_red = fwd_times_y[1] / (fwd_times_y[1] + fwd_times_i[1] + 1e-8)
        fwd_02_blue = fwd_times_g[1] / (fwd_times_g[1] + fwd_times_i[1] + 1e-8)

        # 0.5 and 0.2 ratios (backward)
        bwd_05_red = bwd_times_y[0] / (bwd_times_y[0] + bwd_times_i[0] + 1e-8)
        bwd_05_blue = bwd_times_g[0] / (bwd_times_g[0] + bwd_times_i[0] + 1e-8)
        bwd_02_red = bwd_times_y[1] / (bwd_times_y[1] + bwd_times_i[1] + 1e-8)
        bwd_02_blue = bwd_times_g[1] / (bwd_times_g[1] + bwd_times_i[1] + 1e-8)

        avocado_feats.extend([
            fwd_times_i[0], fwd_times_i[1], fwd_05_red, fwd_05_blue, fwd_02_red, fwd_02_blue,
            bwd_times_i[0], bwd_times_i[1], bwd_05_red, bwd_05_blue, bwd_02_red, bwd_02_blue
        ])

        # Percentile differences
        percentiles = [10, 30, 50, 70, 90]
        all_frac_percentiles = []
        for p in percentiles:
            frac_percentiles = []
            for c in range(C):
                p_val = np.percentile(X_i[:, c], p)
                mx = np.max(X_i[:, c])
                mn = np.min(X_i[:, c])
                frac_percentiles.append((p_val - mn) / (mx - mn + 1e-8))
            all_frac_percentiles.append(np.nanmedian(frac_percentiles))

        avocado_feats.extend([
            all_frac_percentiles[0] - all_frac_percentiles[2], # 10 - 50
            all_frac_percentiles[1] - all_frac_percentiles[2], # 30 - 50
            all_frac_percentiles[3] - all_frac_percentiles[2], # 70 - 50
            all_frac_percentiles[4] - all_frac_percentiles[2]  # 90 - 50
        ])

        # Secondary Peak Fraction
        peak_fracs_2 = []
        for c in range(C):
            flux = X_i[:, c]
            peaks, props = find_peaks(flux, height=np.max(np.abs(flux))/5.0)
            heights = np.sort(props["peak_heights"])[::-1]
            if len(heights) > 1:
                peak_fracs_2.append(heights[1] / heights[0])
            else:
                peak_fracs_2.append(0.0)
        peak_frac_2 = np.nanmedian(peak_fracs_2) if len(peak_fracs_2) > 0 else 0.0
        avocado_feats.append(peak_frac_2)

        # Append avocado feats to main list
        sample_feats.extend(avocado_feats)
        feats.append(sample_feats)

    feats = np.array(feats, dtype=np.float32)
    feats = np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0)
    return feats


def clean(arr):
    return np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)

In [ ]:
hc_train = compute_handcrafted_features(X_train)
hc_test = compute_handcrafted_features(X_test)

print(f"\nHandcrafted train shape: {hc_train.shape}")
print(f"Handcrafted test  shape: {hc_test.shape}")

concat_train_new = clean(np.concatenate([X_train_emb, hc_train], axis=1))
concat_test_new = clean(np.concatenate([X_test_emb, hc_test], axis=1))

/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]



Handcrafted train shape: (2459, 116)
Handcrafted test  shape: (2466, 116)


In [ ]:
classes = np.unique(y_train)
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)
class_weight_dict = dict(zip(classes, weights))
print("class_weight_dict:", class_weight_dict)

seeds = [42, 52, 62] # 52, 62, 72, 82
models = []

for seed in seeds:
    clf = lgb.LGBMClassifier(
        objective="multiclass",
        num_class=len(classes),
        boosting_type="gbdt",
        n_estimators=400,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=1.0,
        class_weight=class_weight_dict,
        random_state=seed,
        n_jobs=-1,
        verbosity=-1,
    )

    clf.fit(concat_train_new, y_train_np)
    models.append(clf)

proba_list = [clf.predict_proba(concat_test_new) for clf in models]
mean_proba = np.mean(proba_list, axis=0)
y_pred = mean_proba.argmax(axis=1)

print("Accuracy:", accuracy_score(y_test_np, y_pred))
print("Balanced Accuracy:", balanced_accuracy_score(y_test_np, y_pred))
print("Macro F1:", f1_score(y_test_np, y_pred, average="macro"))
print(classification_report(y_test_np, y_pred, digits=4, zero_division=0))

class_weight_dict: {np.int64(0): np.float64(1.4279907084785133), np.int64(1): np.float64(0.6505291005291005), np.int64(2): np.float64(0.46100487439070115), np.int64(3): np.float64(2.8329493087557602), np.int64(4): np.float64(25.091836734693878), np.int64(5): np.float64(5.165966386554622), np.int64(6): np.float64(1.1479925303454714), np.int64(7): np.float64(7.636645962732919), np.int64(8): np.float64(0.5611592879963487), np.int64(9): np.float64(2.582983193277311), np.int64(10): np.float64(1.463690476190476), np.int64(11): np.float64(0.22605258319544033), np.int64(12): np.float64(2.2810760667903525), np.int64(13): np.float64(3.4439775910364148)}


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Accuracy: 0.716139497161395
Balanced Accuracy: 0.5257168937954861
Macro F1: 0.5413231593033437
              precision    recall  f1-score   support

           0     0.7158    0.5484    0.6210       124
           1     0.9925    0.9741    0.9832       270
           2     0.5791    0.4791    0.5244       382
           3     0.0000    0.0000    0.0000        63
           4     0.8571    0.8571    0.8571         7
           5     0.6364    0.2000    0.3043        35
           6     0.4839    0.2941    0.3659       153
           7     0.0000    0.0000    0.0000        24
           8     0.7435    0.9073    0.8173       313
           9     0.6667    0.1471    0.2410        68
          10     0.9355    0.9587    0.9469       121
          11     0.6582    0.9048    0.7621       777
          12     0.9375    0.9740    0.9554        77
          13     0.7500    0.1154    0.2000        52

    accuracy                         0.7161      2466
   macro avg     0.6397    0.5257    0.

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
classes = np.unique(y_train_np)
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_np
)
class_weight_dict = dict(zip(classes, weights))
print("class_weight_dict:", class_weight_dict)

sample_weights = np.array([class_weight_dict[y] for y in y_train_np])

seeds = [42] # 52, 62, 72, 82
models = []

for seed in seeds:
    clf = XGBClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.5,
        random_state=seed,
        tree_method='hist',
        eval_metric='mlogloss'
    )

    clf.fit(concat_train_new, y_train_np, sample_weight=sample_weights)
    models.append(clf)

proba_list = [clf.predict_proba(concat_test_new) for clf in models]
mean_proba = np.mean(proba_list, axis=0)
y_pred = mean_proba.argmax(axis=1)

print("Accuracy:", accuracy_score(y_test_np, y_pred))
print("Balanced Accuracy:", balanced_accuracy_score(y_test_np, y_pred))
print("Macro F1:", f1_score(y_test_np, y_pred, average="macro"))
print(classification_report(y_test_np, y_pred, digits=4, zero_division=0))

class_weight_dict: {np.int64(0): np.float64(1.4279907084785133), np.int64(1): np.float64(0.6505291005291005), np.int64(2): np.float64(0.46100487439070115), np.int64(3): np.float64(2.8329493087557602), np.int64(4): np.float64(25.091836734693878), np.int64(5): np.float64(5.165966386554622), np.int64(6): np.float64(1.1479925303454714), np.int64(7): np.float64(7.636645962732919), np.int64(8): np.float64(0.5611592879963487), np.int64(9): np.float64(2.582983193277311), np.int64(10): np.float64(1.463690476190476), np.int64(11): np.float64(0.22605258319544033), np.int64(12): np.float64(2.2810760667903525), np.int64(13): np.float64(3.4439775910364148)}
Accuracy: 0.7124898621248986
Balanced Accuracy: 0.5286190713204333
Macro F1: 0.5446783183149849
              precision    recall  f1-score   support

           0     0.7253    0.5323    0.6140       124
           1     0.9925    0.9741    0.9832       270
           2     0.5490    0.4843    0.5146       382
           3     0.3333    0.0159  

## Testing only avocado

In [ ]:
import numpy as np
import scipy.stats as sp_stats
from scipy.signal import find_peaks

def find_time_to_fractions(fluxes, fractions, forward=True):
    max_time = np.argmax(fluxes)
    max_flux = fluxes[max_time]

    result = np.zeros(len(fractions))
    frac_idx = 0
    offset = 0

    while True:
        offset += 1
        if forward:
            new_time = max_time + offset
            if new_time >= len(fluxes):
                result[frac_idx:] = offset
                break
        else:
            new_time = max_time - offset
            if new_time < 0:
                result[frac_idx:] = offset
                break

        test_flux = fluxes[new_time]
        while test_flux < max_flux * fractions[frac_idx]:
            result[frac_idx] = offset
            frac_idx += 1
            if frac_idx == len(fractions):
                break

        if frac_idx == len(fractions):
            break

    return result


def compute_base_features(X):
    X = np.asarray(X, dtype=np.float32)

    if X.ndim != 3:
        raise ValueError(f"Expected (N, T, C), got {X.shape}")

    N, T, C = X.shape
    feats = []

    for i in range(N):
        sample_feats = []
        X_i = X[i]

        # Per-channel stats
        for c in range(C):
            ts = X_i[:, c]
            sample_feats.extend([
                np.mean(ts),
                np.std(ts),
                np.min(ts),
                np.max(ts),
                np.ptp(ts),  # range
                float(sp_stats.skew(ts)),
                float(sp_stats.kurtosis(ts)),
                np.polyfit(np.arange(T), ts, 1)[0],  # slope
                ts[0],
                ts[-1],
            ])

        # Cross-channel correlations
        for c1 in range(C):
            for c2 in range(c1 + 1, C):
                r = np.corrcoef(X_i[:, c1], X_i[:, c2])[0, 1]
                sample_feats.append(r if np.isfinite(r) else 0.0)

        # Amplitude ratios
        amplitudes = [np.ptp(X_i[:, c]) for c in range(C)]
        for c1 in range(C):
            for c2 in range(c1 + 1, C):
                ratio = amplitudes[c1] / (amplitudes[c2] + 1e-8)
                sample_feats.append(np.log1p(ratio))

        feats.append(sample_feats)

    feats = np.array(feats, dtype=np.float32)
    feats = np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0)
    return feats


def compute_avocado_features(X):
    X = np.asarray(X, dtype=np.float32)

    if X.ndim != 3:
        raise ValueError(f"Expected (N, T, C), got {X.shape}")

    N, T, C = X.shape

    # Define band indices (assuming u=0, g=1, r=2, i=3, z=4, y=5)
    b_g = 1 # blue band
    b_i = 3 # reference band
    b_y = 5 # red band

    feats = []

    for i in range(N):
        sample_feats = []
        X_i = X[i]

        max_flux_i = np.max(X_i[:, b_i])
        min_flux_i = np.min(X_i[:, b_i])
        max_flux_g = np.max(X_i[:, b_g])
        min_flux_g = np.min(X_i[:, b_g])
        max_flux_y = np.max(X_i[:, b_y])
        min_flux_y = np.min(X_i[:, b_y])

        # Magnitude & Flux ratios
        max_mag = -2.5 * np.log10(np.abs(max_flux_i) + 1e-8)
        pos_flux_ratio = max_flux_i / (max_flux_i - min_flux_i + 1e-8)

        max_flux_ratio_red = np.abs(max_flux_y) / (np.abs(max_flux_y) + np.abs(max_flux_i) + 1e-8)
        max_flux_ratio_blue = np.abs(max_flux_g) / (np.abs(max_flux_i) + np.abs(max_flux_g) + 1e-8)

        min_flux_ratio_red = np.abs(min_flux_y) / (np.abs(min_flux_y) + np.abs(min_flux_i) + 1e-8)
        min_flux_ratio_blue = np.abs(min_flux_g) / (np.abs(min_flux_i) + np.abs(min_flux_g) + 1e-8)

        sample_feats.extend([
            max_mag, pos_flux_ratio,
            max_flux_ratio_red, max_flux_ratio_blue,
            min_flux_ratio_red, min_flux_ratio_blue
        ])

        # Timing differences
        max_dt = np.argmax(X_i[:, b_y]) - np.argmax(X_i[:, b_g])
        sample_feats.append(max_dt)

        # Width Integrals
        positive_width = np.sum(np.clip(X_i[:, b_i], 0, None)) / (max_flux_i + 1e-8)
        negative_width = np.sum(np.clip(X_i[:, b_i], None, 0)) / (min_flux_i - 1e-8)
        sample_feats.extend([positive_width, negative_width])

        # Time to fractions of maximum
        fractions = [0.5, 0.2]
        fwd_times_i = find_time_to_fractions(X_i[:, b_i], fractions, forward=True)
        bwd_times_i = find_time_to_fractions(X_i[:, b_i], fractions, forward=False)
        fwd_times_g = find_time_to_fractions(X_i[:, b_g], fractions, forward=True)
        bwd_times_g = find_time_to_fractions(X_i[:, b_g], fractions, forward=False)
        fwd_times_y = find_time_to_fractions(X_i[:, b_y], fractions, forward=True)
        bwd_times_y = find_time_to_fractions(X_i[:, b_y], fractions, forward=False)

        # 0.5 and 0.2 ratios (forward)
        fwd_05_red = fwd_times_y[0] / (fwd_times_y[0] + fwd_times_i[0] + 1e-8)
        fwd_05_blue = fwd_times_g[0] / (fwd_times_g[0] + fwd_times_i[0] + 1e-8)
        fwd_02_red = fwd_times_y[1] / (fwd_times_y[1] + fwd_times_i[1] + 1e-8)
        fwd_02_blue = fwd_times_g[1] / (fwd_times_g[1] + fwd_times_i[1] + 1e-8)

        # 0.5 and 0.2 ratios (backward)
        bwd_05_red = bwd_times_y[0] / (bwd_times_y[0] + bwd_times_i[0] + 1e-8)
        bwd_05_blue = bwd_times_g[0] / (bwd_times_g[0] + bwd_times_i[0] + 1e-8)
        bwd_02_red = bwd_times_y[1] / (bwd_times_y[1] + bwd_times_i[1] + 1e-8)
        bwd_02_blue = bwd_times_g[1] / (bwd_times_g[1] + bwd_times_i[1] + 1e-8)

        sample_feats.extend([
            fwd_times_i[0], fwd_times_i[1], fwd_05_red, fwd_05_blue, fwd_02_red, fwd_02_blue,
            bwd_times_i[0], bwd_times_i[1], bwd_05_red, bwd_05_blue, bwd_02_red, bwd_02_blue
        ])

        # Percentile differences
        percentiles = [10, 30, 50, 70, 90]
        all_frac_percentiles = []
        for p in percentiles:
            frac_percentiles = []
            for c in range(C):
                p_val = np.percentile(X_i[:, c], p)
                mx = np.max(X_i[:, c])
                mn = np.min(X_i[:, c])
                frac_percentiles.append((p_val - mn) / (mx - mn + 1e-8))
            all_frac_percentiles.append(np.nanmedian(frac_percentiles))

        sample_feats.extend([
            all_frac_percentiles[0] - all_frac_percentiles[2], # 10 - 50
            all_frac_percentiles[1] - all_frac_percentiles[2], # 30 - 50
            all_frac_percentiles[3] - all_frac_percentiles[2], # 70 - 50
            all_frac_percentiles[4] - all_frac_percentiles[2]  # 90 - 50
        ])

        # Secondary Peak Fraction
        peak_fracs_2 = []
        for c in range(C):
            flux = X_i[:, c]
            peaks, props = find_peaks(flux, height=np.max(np.abs(flux))/5.0)
            heights = np.sort(props["peak_heights"])[::-1]
            if len(heights) > 1:
                peak_fracs_2.append(heights[1] / heights[0])
            else:
                peak_fracs_2.append(0.0)
        peak_frac_2 = np.nanmedian(peak_fracs_2) if len(peak_fracs_2) > 0 else 0.0
        sample_feats.append(peak_frac_2)

        feats.append(sample_feats)

    feats = np.array(feats, dtype=np.float32)
    feats = np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0)
    return feats

In [ ]:
hc_train = compute_base_features(X_train)
hc_test = compute_base_features(X_test)

print(f"\nHandcrafted train shape: {hc_train.shape}")
print(f"Handcrafted test  shape: {hc_test.shape}")

avocado_train = compute_avocado_features(X_train)
avocado_test = compute_avocado_features(X_test)

print(f"\nAvocado train shape: {avocado_train.shape}")
print(f"Avocado test  shape: {avocado_test.shape}")

concat_base_train = clean(np.concatenate([X_train_emb, hc_train], axis=1))
concat_base_test = np.concatenate([X_test_emb, hc_test], axis=1)

concat_avocado_train = clean(np.concatenate([X_train_emb, avocado_train], axis=1))
concat_avocado_test = clean(np.concatenate([X_test_emb, avocado_test], axis=1))

/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]



Handcrafted train shape: (2459, 90)
Handcrafted test  shape: (2466, 90)

Avocado train shape: (2459, 26)
Avocado test  shape: (2466, 26)


In [ ]:
classes = np.unique(y_train)
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)
class_weight_dict = dict(zip(classes, weights))
print("class_weight_dict:", class_weight_dict)

seeds = [42] # 52, 62, 72, 82
models = []

for seed in seeds:
    clf = lgb.LGBMClassifier(
        objective="multiclass",
        num_class=len(classes),
        boosting_type="gbdt",
        n_estimators=200,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=1.0,
        class_weight=class_weight_dict,
        random_state=seed,
        n_jobs=-1,
        verbosity=-1,
    )

    clf.fit(concat_avocado_train, y_train_np)
    models.append(clf)

proba_list = [clf.predict_proba(concat_avocado_test) for clf in models]
mean_proba = np.mean(proba_list, axis=0)
y_pred = mean_proba.argmax(axis=1)

print("Accuracy:", accuracy_score(y_test_np, y_pred))
print("Balanced Accuracy:", balanced_accuracy_score(y_test_np, y_pred))
print("Macro F1:", f1_score(y_test_np, y_pred, average="macro"))
print(classification_report(y_test_np, y_pred, digits=4, zero_division=0))

class_weight_dict: {np.int64(0): np.float64(1.4279907084785133), np.int64(1): np.float64(0.6505291005291005), np.int64(2): np.float64(0.46100487439070115), np.int64(3): np.float64(2.8329493087557602), np.int64(4): np.float64(25.091836734693878), np.int64(5): np.float64(5.165966386554622), np.int64(6): np.float64(1.1479925303454714), np.int64(7): np.float64(7.636645962732919), np.int64(8): np.float64(0.5611592879963487), np.int64(9): np.float64(2.582983193277311), np.int64(10): np.float64(1.463690476190476), np.int64(11): np.float64(0.22605258319544033), np.int64(12): np.float64(2.2810760667903525), np.int64(13): np.float64(3.4439775910364148)}
Accuracy: 0.6780210867802109
Balanced Accuracy: 0.4499350507887379
Macro F1: 0.46898513783574275
              precision    recall  f1-score   support

           0     0.6533    0.3952    0.4925       124
           1     0.9455    0.9630    0.9541       270
           2     0.5256    0.4293    0.4726       382
           3     0.5000    0.0159 

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


# Classic classif head

In [ ]:
import torch
import torch.nn as nn
from chronos import Chronos2Pipeline

class Chronos2EmbeddingClassifier(nn.Module):
    def __init__(
        self,
        n_classes: int,
        input_channels: int = 6,
        head_hidden_dim: int = 256,
        dropout: float = 0.2,
        chronos_model_name: str = "amazon/chronos-2",
    ):
        super().__init__()
        self.input_channels = input_channels

        chronos_device = "cuda" if torch.cuda.is_available() else "cpu"
        chronos_dtype = torch.bfloat16 if chronos_device == "cuda" else torch.float32

        self.pipeline = Chronos2Pipeline.from_pretrained(
            chronos_model_name,
            device_map=chronos_device,
            torch_dtype=chronos_dtype,
        )

        for param in self.pipeline.model.parameters():
            param.requires_grad = False

        embedding_dim = self.pipeline.model.config.d_model

        pooled_feature_dim = input_channels * 3 * embedding_dim

        self.feature_dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
          nn.Linear(pooled_feature_dim, 1024),
          nn.BatchNorm1d(1024),
          nn.GELU(),
          nn.Dropout(dropout),

          nn.Linear(1024, head_hidden_dim),
          nn.BatchNorm1d(head_hidden_dim),
          nn.GELU(),
          nn.Dropout(dropout),

          nn.Linear(head_hidden_dim, n_classes),
      )

    @torch.no_grad()
    def extract_embeddings(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim != 3:
            raise ValueError(f"Expected (batch, timesteps, channels), got {tuple(x.shape)}")

        x_permuted = x.permute(0, 2, 1).contiguous()

        chronos_inputs = [sample.cpu() for sample in x_permuted]

        embeddings, _ = self.pipeline.embed(chronos_inputs)

        pooled_features = []
        for sample_embedding in embeddings:
            token_mean = sample_embedding.mean(dim=1)
            token_max = sample_embedding.max(dim=1).values
            token_min = sample_embedding.min(dim=1).values

            pooled = torch.cat([token_mean, token_max, token_min], dim=-1).reshape(-1)
            pooled_features.append(pooled)

        return torch.stack(pooled_features, dim=0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.extract_embeddings(x).to(device=x.device, dtype=torch.float32)
        features = self.feature_dropout(features)
        return self.classifier(features)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = Chronos2EmbeddingClassifier(
    n_classes=N_CLASSES,
    input_channels=N_CH,
    head_hidden_dim=256,
    dropout=0.3,
    chronos_model_name="amazon/chronos-2",
).to(device)

xb, yb = next(iter(train_loader))
with torch.no_grad():
    sample_features = model.extract_embeddings(xb)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Chronos-2 pooled feature shape : {sample_features.shape}")
print(f"Trainable head params          : {trainable_params:,}")

Chronos-2 pooled feature shape : torch.Size([512, 13824])
Trainable head params          : 14,425,358


In [ ]:
model, history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=test_loader,
    y_train=y_train,
    device=device,
    n_epochs=20,
    lr=1e-3,
    weight_decay=1e-4,
    patience=10,
    label="Chronos2_embedding_classifier"
)
results_chronos2 = test_model(
    model=model,
    test_loader=test_loader,
    y_train=y_train,
    device=device
)


Training: Chronos2_embedding_classifier
Epoch |   TrLoss |   TrAcc |    TrF1 |   VaLoss |   VaAcc |    VaF1
------------------------------------------------------------------------
    1 |   2.6210 | 22.81% | 0.2063 |   2.8467 | 18.37% | 0.1353  <- best
    2 |   2.0117 | 35.79% | 0.3425 |   2.6665 | 30.62% | 0.2676  <- best
    3 |   1.7201 | 41.20% | 0.4126 |   2.5808 | 36.01% | 0.3231  <- best
    4 |   1.5407 | 48.47% | 0.4913 |   2.4039 | 41.93% | 0.3861  <- best
    5 |   1.3748 | 54.58% | 0.5523 |   2.3552 | 43.84% | 0.3955  <- best
    6 |   1.2662 | 61.85% | 0.6138 |   2.3972 | 44.24% | 0.4030  <- best
    7 |   1.1816 | 66.61% | 0.6705 |   2.3811 | 43.07% | 0.3850
    8 |   1.1233 | 72.02% | 0.7123 |   2.4349 | 43.11% | 0.3738
    9 |   1.0841 | 77.10% | 0.7579 |   2.4124 | 44.36% | 0.4004
   10 |   1.0423 | 80.15% | 0.7923 |   2.3189 | 52.11% | 0.4295  <- best
   11 |   1.0191 | 80.81% | 0.7943 |   2.3169 | 53.28% | 0.4349  <- best
   12 |   1.0046 | 83.86% | 0.8187 |   2.3

# Baseline

In [ ]:
import copy
import time
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn


def train_one_epoch(model, loader, optimizer, criterion, device, max_grad_norm=1.0):
    model.train()
    total_loss = 0.0
    all_preds, all_targets = [], []

    for xb, yb in loader:
        xb = xb.to(device).float()
        yb = yb.to(device).long()

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()

        if max_grad_norm is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

        optimizer.step()

        total_loss += loss.item() * xb.size(0)
        preds = logits.argmax(dim=1)

        all_preds.append(preds.detach().cpu())
        all_targets.append(yb.detach().cpu())

    all_preds = torch.cat(all_preds).numpy()
    all_targets = torch.cat(all_targets).numpy()

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_targets, all_preds)
    f1 = f1_score(all_targets, all_preds, average="macro")

    return avg_loss, acc, f1


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_targets = [], []

    for xb, yb in loader:
        xb = xb.to(device).float()
        yb = yb.to(device).long()

        logits = model(xb)
        loss = criterion(logits, yb)

        total_loss += loss.item() * xb.size(0)
        preds = logits.argmax(dim=1)

        all_preds.append(preds.cpu())
        all_targets.append(yb.cpu())

    all_preds = torch.cat(all_preds).numpy()
    all_targets = torch.cat(all_targets).numpy()

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_targets, all_preds)
    f1 = f1_score(all_targets, all_preds, average="macro")
    report = classification_report(all_targets, all_preds, digits=4)

    return avg_loss, acc, f1, all_preds, all_targets, report

In [ ]:
def train_model(
    model,
    train_loader,
    val_loader,
    y_train,
    device,
    n_epochs=30,
    lr=1e-3,
    weight_decay=1e-4,
    patience=8,
    label="model",
):
    classes = np.unique(y_train)
    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y_train
    )
    weights_tensor = torch.tensor(class_weights, dtype=torch.float32, device=device)

    criterion = nn.CrossEntropyLoss(
        weight=weights_tensor
    )

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr,
        weight_decay=weight_decay
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=2,
    )

    history = {
        "train_loss": [], "train_acc": [], "train_f1": [],
        "val_loss": [], "val_acc": [], "val_f1": []
    }

    best_f1 = -1.0
    best_state = copy.deepcopy(model.state_dict())
    patience_count = 0

    print(f"\n{'='*72}")
    print(f"Training: {label}")
    print(f"{'='*72}")
    print(f"{'Epoch':>5} | {'TrLoss':>8} | {'TrAcc':>7} | {'TrF1':>7} | {'VaLoss':>8} | {'VaAcc':>7} | {'VaF1':>7}")
    print(f"{'-'*72}")

    t0 = time.time()

    for epoch in range(1, n_epochs + 1):
        tr_loss, tr_acc, tr_f1 = train_one_epoch(
            model, train_loader, optimizer, criterion, device
        )
        va_loss, va_acc, va_f1, _, _, _ = evaluate(
            model, val_loader, criterion, device
        )

        scheduler.step(va_f1)

        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["train_f1"].append(tr_f1)
        history["val_loss"].append(va_loss)
        history["val_acc"].append(va_acc)
        history["val_f1"].append(va_f1)

        print(
            f"{epoch:>5} | {tr_loss:>8.4f} | {tr_acc:>6.2%} | {tr_f1:>6.4f} | "
            f"{va_loss:>8.4f} | {va_acc:>6.2%} | {va_f1:>6.4f}",
            end=""
        )

        if va_f1 > best_f1:
            best_f1 = va_f1
            best_state = copy.deepcopy(model.state_dict())
            patience_count = 0
            print("  <- best", end="")
        else:
            patience_count += 1
            if patience_count >= patience:
                print(f"\nEarly stopping at epoch {epoch}.")
                break

        print()

    elapsed = time.time() - t0
    print(f"\nDone | Best val macro-F1 = {best_f1:.4f} | Time = {elapsed:.1f}s")

    model.load_state_dict(best_state)
    return model, history

In [ ]:
def test_model(model, test_loader, y_train, device):
    classes = np.unique(y_train)
    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y_train
    )
    weights_tensor = torch.tensor(class_weights, dtype=torch.float32, device=device)

    criterion = nn.CrossEntropyLoss(
        weight=weights_tensor
    )

    te_loss, te_acc, te_f1, preds, targets, report = evaluate(
        model, test_loader, criterion, device
    )

    print("\nTest results")
    print(f"Loss     : {te_loss:.4f}")
    print(f"Accuracy : {te_acc:.4%}")
    print(f"Macro-F1 : {te_f1:.4f}")
    print("\nClassification report:\n")
    print(report)

    return {
        "test_loss": te_loss,
        "test_acc": te_acc,
        "test_f1": te_f1,
        "preds": preds,
        "targets": targets,
        "report": report,
    }

In [ ]:
class CNN1DClassifier(nn.Module):

    def __init__(self, n_channels: int, n_steps: int, n_classes: int):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(n_channels, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64), nn.GELU(),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128), nn.GELU(),
            nn.AdaptiveAvgPool1d(8),
            nn.Conv1d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256), nn.GELU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        return self.head(self.conv(x.permute(0, 2, 1)))


In [ ]:
model_cnn = CNN1DClassifier(N_CH, N_STEPS, N_CLASSES).to(device)
total_cnn = sum(p.numel() for p in model_cnn.parameters())
print(f"CNN baseline params : {total_cnn:,}")

model_cnn, history_cnn = train_model(
    model=model_cnn,
    train_loader=train_loader,
    val_loader=test_loader,
    y_train=y_train,
    device=device,
    n_epochs=50,
    lr=1e-3,
    patience=12,
    label="1D_CNN"
)

In [ ]:
results_cnn = test_model(
    model=model_cnn,
    test_loader=test_loader,
    y_train=y_train,
    device=device
)